<a href="https://colab.research.google.com/github/miso-20/ESSA/blob/main/ESAA_OB_WEEK_05_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

파이썬 머신러닝 완벽 가이드 ch9. p.625~647

## 07. 행렬 분해를 이용한 잠재 요인 협업 필터링 실습

행렬 분해 잠재 요인 협업 필터링: SVD나 NMF 등을 적용할 수 있는데, 일반적으로 행렬 분해에는 SVD가 자주 사용

but, 사용자-아이템 평점 행렬에는 사용자가 평점을 매기지 않은 널 데이터가 많기 때문에 주로 SGD나 ALS 기반의 행렬 분해를 이용


In [ ]:
### 추가 ###
import numpy as np

def get_rmse(R, P, Q, non_zeros):
  error = 0
  for i, j, r in non_zeros:
    pred = np.dot(P[i, :], Q[j, :].T)
    error += pow(r - pred, 2)
    rmse = np.sqrt(error / len(non_zeros))

    return rmse

In [ ]:
def matrix_factorization(R, K, steps=200, learning_rate=0.01, r_lambda = 0.01 ):
  num_users, num_items = R.shape
  # P와 Q 매트릭스의 크기를 지정하고 정규 분포를 가진 랜덤한 값으로 입력합니다.
  np.random.seed(1)
  P = np.random.normal(scale=1./K, size=(num_users, K))
  Q = np.random.normal(scale=1./K, size=(num_items, K))

  # R 〉0 인 행 위치, 열 위치, 값을 non.zeros 리스트 객체에 저장.
  non_zeros = [ (i, j, R[i, j]) for i in range(num_users) for j in range(num_items) if R[i, j] > 0 ]

  # SGD기법으로 P와 Q 매트릭스를 계속 업데이트.
  for step in range(steps):
    for i, j, r in non_zeros:
      # 실제 값과 예측 값의 차이인 오류 값 구함
      eij = r - np.dot(P[i,:], Q[j,:].T)
      # Regularization을 반영한 SGD 업데이트 공식 적용
      P[i, :] = P[i, :] + learning_rate*(eij * Q[j, :] - r_lambda*P[i, :])
      Q[j, :] = Q[j, :] + learning_rate*(eij * P[i, :] - r_lambda*Q[j, :])

    rmse = get_rmse(R, P, Q, non_zeros)
    if (step % 10) == 0:
      print("### iteration step : ", step, " rmse : ", rmse)
  return P, Q

In [ ]:
import pandas as pd
import numpy as np
movies = pd.read_csv('/content/drive/MyDrive/ESAA_OB/OB_data/movies.csv')
ratings = pd.read_csv('/content/drive/MyDrive/ESAA_OB/OB_data/ratings.csv')
ratings = ratings[['userId', 'movieId', 'rating']]
ratings_matrix = ratings.pivot_table('rating', index='userId', columns='movieId')

# title 칼럼을 얻기 위해 movies와 조인 수행
rating_movies = pd.merge(ratings, movies, on='movieId')
# columns='title' 로 title 칼럼으로 pivot 수행.
ratings_matrix = rating_movies.pivot_table('rating', index='userId', columns='title')

In [ ]:
P, Q = matrix_factorization(ratings_matrix.values, K=50, steps=200, learning_rate=0.01,
                            r_lambda = 0.01)
pred_matrix = np.dot(P, Q.T)

### iteration step :  0  rmse :  0.012610190185352439
### iteration step :  10  rmse :  0.0013061177073945894
### iteration step :  20  rmse :  0.0008993378168068016
### iteration step :  30  rmse :  0.0007321609715871822
### iteration step :  40  rmse :  0.0004910959462488221
### iteration step :  50  rmse :  0.0003754508200780458
### iteration step :  60  rmse :  0.00034855767131719643
### iteration step :  70  rmse :  0.0003561595232345271
### iteration step :  80  rmse :  0.0003708624777264578
### iteration step :  90  rmse :  0.0003819741639314076
### iteration step :  100  rmse :  0.00038660396173307416
### iteration step :  110  rmse :  0.00038506818860388335
### iteration step :  120  rmse :  0.0003788144389231676
### iteration step :  130  rmse :  0.0003694586889838778
### iteration step :  140  rmse :  0.00035836938647198927
### iteration step :  150  rmse :  0.0003465497794673754
### iteration step :  160  rmse :  0.0003346645008794443
### iteration step :  170  rmse :  0.00

In [ ]:
ratings_pred_matrix = pd.DataFrame(data=pred_matrix, index= ratings_matrix.index,
                                   columns = ratings_matrix.columns)
ratings_pred_matrix.head(3)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,3.055084,4.092018,3.564130,4.502167,3.981215,1.271694,3.603274,2.333266,5.091749,3.972454,...,1.402608,4.208382,3.705957,2.720514,2.787331,3.475076,3.253458,2.161087,4.010495,0.859474
2,3.170119,3.657992,3.308707,4.166521,4.311890,1.275469,4.237972,1.900366,3.392859,3.647421,...,0.973811,3.528264,3.361532,2.672535,2.404456,4.232789,2.911602,1.634576,4.135735,0.725684
3,2.307073,1.658853,1.443538,2.208859,2.229486,0.780760,1.997043,0.924908,2.970700,2.551446,...,0.520354,1.709494,2.281596,1.782833,1.635173,1.323276,2.887580,1.042618,2.293890,0.396941


In [ ]:
### 추가 ###
# 사용자 9번 영화추천
def get_unseen_movies(ratings_matrix, userId):
  # userId로 입력받은 사용자의 모든 영화 정보를 추출해 Series로 반환함.
  # 반환된 user_rating은 영화명(title)을 인덱스를 가지는 Series 객체임.
  user_rating = ratings_matrix.loc[userId, :]

  #user_rating이 0보다 크면 기존에 관람한 영화임. 대상 인덱스를 추출해 list 객체로 만듦.
  already_seen = user_rating[ user_rating > 0].index.tolist()

  # 모든 영화명을 list 객체로 만듦.
  movies_list = ratings_matrix.columns.tolist()

  # list comporehension으로 already_seen에 해당하는 영화는 movies_list에서 제외함.
  unseen_list = [ movie for movie in movies_list if movie not in already_seen]

  return unseen_list

def recomm_movie_by_userId(pred_df, userId, unseen_list, top_n = 10):
  # 예측 평점 DataFrame에서 사용자id 인덱스와 unseen_list로 들어온 영화명 칼럼을 추출해
  # 가장 예측 평점이 높은 순으로 정렬함.
  recomm_movies = pred_df.loc[userId, unseen_list].sort_values(ascending=False)[:top_n]
  return recomm_movies

In [ ]:
# 사용자가 관람하지 않은 영화명 추출
unseen_list = get_unseen_movies(ratings_matrix, 9)

# 잠재 요인 협업 필터링으로 영화 추천
recomm_movies = recomm_movie_by_userId(ratings_pred_matrix, 9, unseen_list, top_n=10)

# 평점 데이터를 DataFrame으로 생성.
recomm_movies = pd.DataFrame(data=recomm_movies.values, index=recomm_movies.index,
                             columns=['pred_score'])
recomm_movies

,pred_score
title,
Rear Window (1954),5.704612
"South Park: Bigger, Longer and Uncut (1999)",5.451100
Rounders (1998),5.298393
Blade Runner (1982),5.244951
Roger & Me (1989),5.191962
Gattaca (1997),5.183179
Ben-Hur (1959),5.130463
Rosencrantz and Guildenstern Are Dead (1990),5.087375
"Big Lebowski, The (1998)",5.038690


## 08. 파이썬 추천 시스템 패키지 - Surprise

### 1) Surprise 패키지 소개

In [ ]:
!pip install scikit-surprise

Surprise 패키지

- 파이썬 기반의 추천 시스템 구축을 위한 전용 패키지 중의 하나

- 파이썬 기반에서 사 이킷런과 유사한 API와 프레임워크를 제공

- 추천 시스템의 전반적인 알고리즘을 이해하 고 사이킷런 사용 경험이 있으면 쉽게 사용할 수 있음

주요 장점

- 다양한 추천 알고리즘, 예를 들어 사용자 또는 아이템 기반 최근접 이웃 협업 필터링, SVD, SVD++, NMF 기반의 잠재 요인 협업 필터링을 쉽게 적용해 추천 시스템을 구축할 수 있음

- Surprise의 핵심 API는 사이킷런의 핵심 API와 유사한 API명으로 작성됐음.

  예를 들어 fit( ), predict( ) AP로 추천 데이터 학습과 예측, train_test_split( )으로 추천 학습 데이터 세트와 예측 데이터 세트 분리, cross_validate( ), GridSearchCV 클래스를 통해 추천 시스템을 위한 모델 셀렉션, 평가, 하이퍼 파라미터 튜닝 등의 기능을 제공함


### 2) Surprise를 이용한 추천 시스템 구축

In [ ]:
from surprise import SVD
from surprise import Dataset
from surprise import accuracy
from surprise.model_selection import train_test_split

Surprise
- Movie Lens 데이터 세트의 사용자-영화 평점 데이터 포맷과 같이 userId(사용자 ID), movield(영화 ID), rating(평점)과 같은 주요 데이터가 로우(Row) 레벨 형태로 돼 있는 포맷의 데이터만 처리함

Surprise에 사용자-아이템 평점 데이터를 적용할 때 주의해야 할 점
- 무비렌즈 사이트에서 내려받은 데이터 파일과 동일하게 로우 레벨의 사용자-아이템 평점 데이터를 그대로 적용해야 함

In [ ]:
data = Dataset.load_builtin('ml-100k')
# 수행 시마다 동일하게 데이터를 분할하기 위해 random.state 값 부여
trainset, testset = train_test_split(data, test_size=.25, random_state=0)

In [ ]:
algo = SVD(random_state=0)
algo.fit(trainset)

Surprise에서 추천을 예측하는 메서드는 test()와 predict(), 두 개.

- test(): 사용자-아이템 평점 데이터 세 트 전체에 대해서 추천을 예측하는 메서드.
  
  즉, 입력된 데이터 세트에 대해 추천 데이터 세트를 만들어 줌

- predict(): 개별 사용자와 영화에 대한 추천 평점을 반환해 줌

In [ ]:
predictions = algo.test( testset )
print('prediction type :', type(predictions), ' size:', len(predictions))
print('prediction 결과의 최초 5개 추출')
predictions[:5]

prediction type : <class 'list'>  size: 25000
prediction 결과의 최초 5개 추출


[Prediction(uid='120', iid='282', r_ui=4.0, est=np.float64(3.5114147666251547), details={'was_impossible': False}),
 Prediction(uid='882', iid='291', r_ui=4.0, est=np.float64(3.573872419581491), details={'was_impossible': False}),
 Prediction(uid='535', iid='507', r_ui=5.0, est=np.float64(4.033583485472447), details={'was_impossible': False}),
 Prediction(uid='697', iid='244', r_ui=5.0, est=np.float64(3.8463639495936905), details={'was_impossible': False}),
 Prediction(uid='751', iid='385', r_ui=4.0, est=np.float64(3.1807542478219157), details={'was_impossible': False})]

In [ ]:
[ (pred.uid, pred.iid, pred.est) for pred in predictions[:3] ]

[('120', '282', np.float64(3.5114147666251547)),
 ('882', '291', np.float64(3.573872419581491)),
 ('535', '507', np.float64(4.033583485472447))]

In [ ]:
# 사용자 아이디, 아이템 아이디는 문자열로 입력해야 함.
uid = str(196)
iid = str(302)
pred = algo.predict(uid, iid)
print(pred)

user: 196        item: 302        r_ui = None   est = 4.49   {'was_impossible': False}


In [ ]:
accuracy.rmse(predictions)

RMSE: 0.9467


np.float64(0.9466860806937948)

### 3) Surprise 주요 모듈 소개

**Dataset**

<img src="https://drive.google.com/uc?export=view&id=181E_j0jGYPjFpzGQ88QNzOkX7eS3RXgm" width="600">


<img src="https://drive.google.com/uc?export=view&id=1EKbsxDq7pdGjNOVn-EkPYlt9LMedarps" width="600">


**OS 파일 데이터를 Surprise 데이터 세트로 로딩**

Surprise에 OS 파일을 로딩할 때의 주의할 점

- 로딩되는 데이터 파일에 칼럼명을 가지는 헤더 문자열이 있어서는 안 됨



In [ ]:
import pandas as pd

ratings = pd.read_csv('/content/drive/MyDrive/ESAA_OB/OB_data/ratings.csv')
# ratings_noh.csv 파일로 언로드 시 인덱스와 헤더를 모두 제거한 새로운 파일 생성.
ratings.to_csv('/content/drive/MyDrive/ESAA_OB/OB_data/ratings_noh.csv', index=False, header=False)

- ratings_noh.csv 파일: ratings.csv 파일에서 헤더가 삭제된 파일

- Reader 클래스: 로딩될 ratings_noh.csv 파일의 파싱 정보를 알려주기 위해 사용됨

- ratings_noh.csv: 칼럼 헤더가 없고, 4개의 칼럼이 콤마로만 분리돼 있음

  이 4개의 칼럼이 사용자 아이디, 아이템 아이디, 평점, 타임스탬프임을 로딩할 때 알려줘야 함



In [ ]:
from surprise import Reader

reader = Reader(line_format='user item rating timestamp', sep=',', rating_scale=(0.5, 5))
data=Dataset.load_from_file('/content/drive/MyDrive/ESAA_OB/OB_data/ratings_noh.csv', reader=reader)

Reader 클래스의 주요 생성 파라미터

- line_format (string): 칼럼을 순서대로 나열. 입력된 문자열을 공백으로 분리해 칼럼으로 인식

- sep (char): 칼럼을 분리하는 분리자이며, 디폴트는 '\t'. 판다스 DataFrame에서 입력받을 경우에는 기재할 필요가 없음

- rating_scale (tuple, optional): 평점 값의 최소 ~ 최대 평점을 설정. 디폴트는 (1, 5)이지만 ratings.csv 파일의 경우는 최소 평점이 0.5, 최대 평점이 5이므로 (0.5, 5)로 설정했음.

In [ ]:
trainset, testset = train_test_split(data, test_size=.25, random_state=0)

# 수행 시마다 동일한 결과를 도출하기 위해 random_state 설정
algo = SVD(n_factors=50, random_state=0)

# 학습 데이터 세트로 학습하고 나서 테스트 데이터 세트로 평점 예측 후 RMSE 평가
algo.fit(trainset)
predictions = algo.test( testset )
accuracy.rmse(predictions)

RMSE: 0.8682


np.float64(0.8681952927143516)

**판다스 DataFrame에서 Surprise 데이터 세트로 로딩**

Dataset.load_from_df( )를 이용하면 판다스의 DataFrame에서도 Surprise 데이터 세트로 로딩할 수 있음

- 주의할 점: DataFrame 역시 사용자 아이디, 아이템 아이디, 평점 칼럼 순서를 지켜야 함

In [ ]:
import pandas as pd
from surprise import Reader, Dataset

ratings = pd.read_csv('/content/drive/MyDrive/ESAA_OB/OB_data/ratings.csv')
reader = Reader(rating_scale=(0.5, 5.0))

# ratings DataFrame에서 칼럼은 사용자 아이디, 아이템 아이디, 평점 순서를 지켜야 합니다.
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=.25, random_state=0)

algo = SVD(n_factors=50, random_state=0)
algo.fit(trainset)
predictions = algo.test( testset )
accuracy.rmse(predictions)

RMSE: 0.8682


np.float64(0.8681952927143516)

### 4) Surprise 추천 알고리즘 클래스

<img src="https://drive.google.com/uc?export=view&id=1PdwUofeEIwOOyBqVQ-ZVfiQQsaAuhx0-" width="600">

Surprise SVD의 비용 함수: 사용자 베이스라인(Baseline) 편향성을 감안한 평점 예측에 Regularization을 적용한 것

<img src="https://drive.google.com/uc?export=view&id=1FZsIF9pmhNCHPdnVZrSMJONETtfJ14Do" width="400">

SVD 클래스의 입력 파라미터

<img src="https://drive.google.com/uc?export=view&id=104JS9tVFyLmk3zjMK1r5Fr2xpF9B1wv6" width="600">



### 5) 베이스라인 평점

베이스라인 평점 (Baseline Rating): 개인의 성향을 반영해 아이템 평가에 편향성(bias) 요소를 반영하여 평점을 부과하는 것

보통 베이스라인 평점은 전체 평균 평점 + 사용자 편향 점수 + 아이템 편향 점수 공식으로 계산

- 전체 평균 평점 = 모든 사용자의 아이템에 대한 평점을 평균한 값

- 사용자 편향 점수 = 사용자별 아이템 평점 평균 값 - 전체 평균 평점

- 아이템 편향 점수 = 아이템별 평점 평균 값 - 전체 평균 평점

### 6) 교차 검증과 하이퍼 파라미터 튜닝

In [ ]:
from surprise.model_selection import cross_validate

# 판다스 DataFrame에서 Surprise 데이터 세트로 데이터 로딩
ratings = pd.read_csv('/content/drive/MyDrive/ESAA_OB/OB_data/ratings.csv') # reading data in pandas df
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
algo = SVD(random_state=0)
cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8688  0.8809  0.8679  0.8758  0.8726  0.8732  0.0048  
MAE (testset)     0.6697  0.6765  0.6686  0.6715  0.6695  0.6712  0.0028  
Fit time          1.23    1.21    1.23    1.54    1.78    1.40    0.23    
Test time         0.11    0.14    0.11    0.19    0.41    0.19    0.11    


{'test_rmse': array([0.86875695, 0.88092589, 0.86790862, 0.87578182, 0.87258328]),
 'test_mae': array([0.66974362, 0.67645239, 0.66859186, 0.67147973, 0.6695167 ]),
 'fit_time': (1.2302169799804688,
  1.2106125354766846,
  1.2276930809020996,
  1.5415706634521484,
  1.7804162502288818),
 'test_time': (0.11125373840332031,
  0.1405632495880127,
  0.11328887939453125,
  0.1868140697479248,
  0.4105498790740967)}

In [ ]:
from surprise.model_selection import GridSearchCV

# 최적화할 파라미터를 딕셔너리 형태로 지정.
param_grid = {'n_epochs': [20, 40, 60], 'n_factors': [50, 100, 200] }

# CV를 3개 폴드 세트로 지정, 성능 평가는 rmse, mse로 수행하도록 GridSearchCV 구성
gs = GridSearchCV(SVD, param_grid, measures=['rmse', 'mae'], cv=3)
gs.fit(data)

from surprise.model_selection import GridSearchCV

# 최고 RMSE Evaluation 점수와 그때의 하이퍼 파라미터
print(gs.best_score['rmse'])
print(gs.best_params['rmse'])

0.8776388808022743
{'n_epochs': 20, 'n_factors': 50}


### 7) Surprise를 이용한 개인화 영화 추천 시스템 구축

In [ ]:
# 다음 코드는 train_test_split( )으로 분리되지 않는 데이터 세트에 fit( )을 호출해 오류가 발생합니다.
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)
algo = SVD(n_factors=50, random_state=0)
algo.fit(data)

AttributeError: 'DatasetAutoFolds' object has no attribute 'n_users'

In [ ]:
from surprise.dataset import DatasetAutoFolds

reader = Reader(line_format='user item rating timestamp', sep=',', rating_scale=(0.5, 5))
# DatasetAutoFolds 클래스를 ratings_noh.csv 파일 기반으로 생성.
data_folds = DatasetAutoFolds(ratings_file='/content/drive/MyDrive/ESAA_OB/OB_data/ratings_noh.csv', reader=reader)

# 전체 데이터를 학습 데이터로 생성함.
trainset = data_folds.build_full_trainset()

In [ ]:
algo = SVD(n_epochs=20, n_factors=50, random_state=0)
algo.fit(trainset)

In [ ]:
# 영화에 대한 상세 속성 정보 DataFrame 로딩
movies = pd.read_csv('/content/drive/MyDrive/ESAA_OB/OB_data/movies.csv')

# userId=9의 movieId 데이터를 추출해 movieId=42 데이터가 있는지 확인.
movieIds = ratings[ratings['userId']==9] ['movieId']
if movieIds[movieIds==42].count() == 0:
  print('사용자 아이디 9는 영화 아이디 42의 평점 없음')

print(movies[movies['movieId']==42])

사용자 아이디 9는 영화 아이디 42의 평점 없음
    movieId                   title              genres
38       42  Dead Presidents (1995)  Action|Crime|Drama


In [ ]:
uid = str(9)
iid = str(42)

pred = algo.predict(uid, iid, verbose=True)

user: 9          item: 42         r_ui = None   est = 3.13   {'was_impossible': False}


In [ ]:
def get_unseen_surprise(ratings, movies, userId):
  # 입력값으로 들어온 userid에 해당하는 사용자가 평점을 매긴 모든 영화를 리스트로 생성
  seen_movies = ratings[ratings['userId']== userId]['movieId'].tolist()
  # 모든 영화의 movield를 리스트로 생성.
  total_movies = movies['movieId'].tolist()

  # 모든 영화의 movield 중 이미 평점을 매긴 영화의 movield를 제외한 후 리스트로 생성
  unseen_movies= [movie for movie in total_movies if movie not in seen_movies]
  print('평점 매긴 영화 수:', len(seen_movies), '추천 대상 영화 수:', len(unseen_movies),
        '전체 영화 수:', len(total_movies))

  return unseen_movies

unseen_movies = get_unseen_surprise(ratings, movies, 9)

평점 매긴 영화 수: 46 추천 대상 영화 수: 9696 전체 영화 수: 9742


In [ ]:
def recomm_movie_by_surprise(algo, userId, unseen_movies, top_n=10):

  # 알고리즘 객체의 predict() 메서드를 평점이 없는 영화에 반복 수행한 후 결과를 list 객체로 저장
  predictions = [algo.predict(str(userId), str(movieId)) for movieId in unseen_movies]

  # predictions list 객체는 surprise의 Predictions 객체를 원소로 가지고 있음.
  # [Prediction(uid='9', iid='1 est=3.69), Prediction(uid='9', iid='2', est=2.98),,,,]

  # 이를 est 값으로 정렬하기 위해서 아래의 sortkey_est 함수를 정의함.
  # sortkey.est 함수는 list 객체의 sort() 함수의 키 값으로 사용되어 정렬 수행.
  def sortkey_est(pred):
    return pred.est

  # sortkey_est( ) 반환값의 내림 차순으로 정렬 수행하고 top_n개의 최상위 값 추출.
  predictions.sort(key=sortkey_est, reverse=True)
  top_predictions= predictions[:top_n]

  # top_n으로 추출된 영화의 정보 추출. 영화 아이디, 추천 예상 평점, 제목 추출
  top_movie_ids = [ int(pred.iid) for pred in top_predictions]
  top_movie_rating = [ pred.est for pred in top_predictions]
  top_movie_titles = movies[movies.movieId.isin(top_movie_ids)]['title']

  top_movie_preds = [ (id, title, rating) for id, title, rating in
                       zip(top_movie_ids, top_movie_titles, top_movie_rating)]

  return top_movie_preds

unseen_movies = get_unseen_surprise(ratings, movies, 9)
top_movie_preds = recomm_movie_by_surprise(algo, 9, unseen_movies, top_n=10)

print('##### Top-10 추천 영화 리스트 #####')
for top_movie in top_movie_preds:
  print(top_movie[1], ":", top_movie[2])

평점 매긴 영화 수: 46 추천 대상 영화 수: 9696 전체 영화 수: 9742
##### Top-10 추천 영화 리스트 #####
Usual Suspects, The (1995) : 4.306302135700814
Star Wars: Episode IV - A New Hope (1977) : 4.281663842987387
Pulp Fiction (1994) : 4.278152632122759
Silence of the Lambs, The (1991) : 4.226073566460876
Godfather, The (1972) : 4.1918097904381995
Streetcar Named Desire, A (1951) : 4.154746591122657
Star Wars: Episode V - The Empire Strikes Back (1980) : 4.122016128534504
Star Wars: Episode VI - Return of the Jedi (1983) : 4.108009609093436
Goodfellas (1990) : 4.083464936588478
Glory (1989) : 4.07887165526957


## 09. 정리

추천 시스템
- 기업 애플리케이션에서 매우 중요한 위치를 차지하고 있음
- 대표적인 방식으로 콘텐츠 기반 필터링과 협업 필터링

    - 콘텐츠 기반 필터링: 아이템(상품, 영화, 서비스 등)을 구성하는 여러 가지 콘텐츠 중 사용자가 좋아하는 콘텐츠를 필터링하여 이에 맞는 아이템을 추천하는 방식

    - 협업 필터링: 최근접 이웃 협업 필터링과 잠재 요인 협업 필터링으로 나뉨.

      - 최근접 이웃 협업 필터링: 다시 사용자 기반(사용자-사용자)과 아이템 기반(아이템-아이템)으로 나뉘며, 이중 아이템 기반이 더 많이 사용
          - 아이템 기반 최근접 이웃 방식: 특정 아이템과 가장 근접하게 유사한 다른 아이템들을 추천하는 방식

      - 잠재 요인 협업 필터링: 많은 추천 시스템에서 활용하는 방식. 사용자-아이템 평점 행렬 데이터에 숨어 있는 잠재 요인을 추출하여 사용자가 아직 평점을 매기지 않은 아이템에 대한 평점을 예측하여 이를 추천에 반영하는 방식


행렬 분해
- 잠재 요인을 추출하기 위해서 다차원의 사용자-아이템 평점 행렬을 저차원의 사용자-잠재요인, 아이템-잠재요인 행렬로 분해하는 기법

Surprise
- 파이썬의 추천 시스템 패키지 중 하나

- 사이킷런과 유사한 API를 지향하며, 간단한 API만을 이용해 파이썬 기반에서 추천 시스템을 구현해 줌
